In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 1. Initialize LLM
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

In [ ]:
class BlogState(TypedDict):
    topic:str
    outline: str
    blog_content: str

In [ ]:
def generate_outline(state:BlogState) -> BlogState:
    topic = state['topic']
    prompt = f"create a detailed outline based on this topic: {topic}"
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

def generate_blog(state:BlogState) -> BlogState:
    outline = state['outline']
    prompt = f"Based on this outline, create a detailed blog: {outline}"
    blog_content = model.invoke(prompt).content
    state['blog_content'] = blog_content
    return state

In [ ]:
graph = StateGraph(BlogState)

graph.add_node("generate_outline",generate_outline)
graph.add_node("generate_blog",generate_blog)

graph.add_edge(START,"generate_outline")
graph.add_edge("generate_outline","generate_blog")
graph.add_edge("generate_blog",END)

workflow = graph.compile()

initial_state = {"topic":"Best Agentic AI tools for coding in 2026"}

final_state = workflow.invoke(initial_state)
print(f"The outline is this: {final_state['outline'].content}")
print(f"The Blog is this: {final_state['blog_content'].content}")

In [ ]:
print(final_state['outline'])
print(final_state['blog_content'])